In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [40]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
import bayesflow as bf

In [42]:
from simulations.benchmarks.hf import hf
from simulations.molecules import MoleculeSimulator
from solvers.evc_solver import EVCSolver

In [43]:
from src.utils.procrustes_utils import (
    compute_procrustes_matrices,
    compute_cc_with_procrustes
)
from src.utils.matrix_utils import orthonormalize_ts

## Simulator

In [44]:
hf_simulator = MoleculeSimulator(
    molecule_fun=hf,
    basis="cc-pVTZ",
    coord_scale=0.1,
    verbose=0,
)

### Reference molecule

In [45]:
include_kwargs = {
    "include_integrals": True,
    "include_hartree_fock": True,
    "include_cc": False,
    "include_coordinates": False,
    "include_all": False
}

hf_reference = hf_simulator.simulate(molecule_kwargs={"bond_distance": 1.75, "perturb": False}, **include_kwargs)

In [46]:
hf_reference["positions"]

array([[0.  , 0.  , 0.  ],
       [1.75, 0.  , 0.  ]], dtype=float32)

In [47]:
reference_determinant = hf_reference["determinant"]
reference_determinant.shape

(44, 44)

In [48]:
reference_overlap = hf_reference["overlaps"]
reference_overlap.shape

(44, 44)

### Sample and target molecules

In [49]:
target_molecules = hf_simulator.sample(10, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 10/10 [00:00<00:00, 632.07it/s]


In [51]:
for k, v in target_molecules.items():
    print(f"{k}: {v.shape}")

atoms: (10, 2)
positions: (10, 2, 3)


In [52]:
target_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=target_molecules["atoms"],
    batched_positions=target_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 10/10 [00:14<00:00,  1.47s/it]


In [53]:
for k, v in target_procrustes_matrices.items():
    print(f"{k}: {v.shape}")

rotation_matrices: (10, 44, 44)
procrustes_orbitals: (10, 44, 44)


In [54]:
sample_molecules = hf_simulator.sample(81, include_kwargs={"include_all": False})

Generating samples: 100%|██████████| 81/81 [00:00<00:00, 374.93it/s]


In [55]:
sample_procrustes_matrices = compute_procrustes_matrices(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing procrustes: 100%|██████████| 81/81 [01:54<00:00,  1.42s/it]


In [56]:
# This replaces setup_sample().
procrustes_cc = compute_cc_with_procrustes(
    batched_atoms=sample_molecules["atoms"],
    batched_positions=sample_molecules["positions"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap,
)

Computing CCSD: 100%|██████████| 81/81 [14:00<00:00, 10.38s/it]


In [57]:
procrustes_cc.keys()

dict_keys(['t1', 't2', 'ccsd_energy'])

In [58]:
for k, v in procrustes_cc.items():
    print(f"{k}: {v.shape}")

t1: (81, 5, 39)
t2: (81, 5, 5, 39, 39)
ccsd_energy: (81,)


In [59]:
evc_solver = EVCSolver(
    geometries=sample_molecules,
    t1s=procrustes_cc["t1"],
    t2s=procrustes_cc["t2"],
    reference_determinant=reference_determinant,
    reference_overlap=reference_overlap
)

In [60]:
orthogonalized_ts = orthonormalize_ts(t1s=evc_solver.t1s, t2s=evc_solver.t2s, lowdin=True)

In [61]:
for k, v in orthogonalized_ts.items():
    print(f"{k}: {v.shape}")

t1: (81, 5, 39)
t2: (81, 5, 5, 39, 39)
coefficients: (81, 81)


In [64]:
orthogonalized_ts["t2"][0].reshape(-1).shape

(38025,)